# Генерация: сборка корпусов и описание

In [51]:
import re
from functools import lru_cache

import numpy as np
import pandas as pd
import pymorphy3

from src.paths import DATA

morph = pymorphy3.MorphAnalyzer()

GEN = DATA / "generation"
RUN_DIR = GEN / "runs" / "main"
FINAL = GEN / "final"
FINAL.mkdir(parents=True, exist_ok=True)

LEVELS = ["level1", "level2", "level3"]
VERSIONS = ["orig"] + LEVELS
CORPORA = ["yt", "tg", "rr"]

In [52]:
# Общеупотребительные сокращения обычных слов

EXPANSIONS = {
    "щас": "сейчас", "ща": "сейчас", "че": "что", "чё": "что", "шо": "что",
    "чето": "что-то", "чёт": "что-то", "оч": "очень", "норм": "нормально",
    "ваще": "вообще", "вообщем": "в общем", "крч": "короче", "тока": "только",
    "типо": "типа", "тоесть": "то есть", "тк": "так как", "пж": "пожалуйста",
    "плз": "пожалуйста", "спс": "спасибо", "кмк": "как мне кажется"
}

WORD = re.compile(r"[А-Яа-яЁё]+")
CAPS = re.compile(r"(?<![А-Яа-яЁё])[А-ЯЁ]{3,}(?![А-Яа-яЁё])")

PHENOMENA = {
    "капс": CAPS.pattern,
    "удлиненные": r"([А-Яа-яЁё])\1{2,}",
    "ряд знаков": r"[!?]{2,}|\.{3,}",
    "эмодзи": "[\U0001F300-\U0001FAFF\u2600-\u27BF]",
    "смайлы": r"[:;=][-^]?[)(DdPp]|\)\)+|\(\(+",
    "числа": r"\d",
    "отрицания": r"\b(не|ни|нет|нельзя|никогда|никто|ничего)\b",
    "сокращения": r"\b(" + "|".join(EXPANSIONS) + r")\b",
    "обращение на ты": r"\b(ты|тебя|тебе|твой|твоя|тво[ие])\b",
    "вопрос": r"\?"
}

has = lambda text, pattern: bool(re.search(pattern, str(text)))

In [53]:
def levenshtein(a, b):
    if abs(len(a) - len(b)) > 4:
        return 9
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def tokens(text):
    return [w.lower().replace("ё", "е") for w in WORD.findall(str(text))]


@lru_cache(maxsize=None)
def lemma(word):
    return morph.parse(word)[0].normal_form.replace("ё", "е")


def lemmas(text):
    return {lemma(w) for w in tokens(text)}


def replacements(before, after):
    gone = lemmas(before) - lemmas(after)
    added = lemmas(after) - lemmas(before)

    allowed = set()
    low = str(before).lower()
    for short, full in EXPANSIONS.items():
        if re.search(rf"\b{short}\b", low):
            allowed |= lemmas(full)

    result = []
    for word in added - allowed:
        if len(word) <= 3:
            continue
        limit = 2 if len(word) <= 5 else 4
        if not any(levenshtein(word, old) <= limit for old in gone):
            result.append(word)
    return result


def caps_split(text):
    abbreviations, emphatic = [], []
    for word in CAPS.findall(str(text)):
        low = word.lower()
        tag = morph.parse(low)[0].tag
        if (not morph.word_is_known(low) or "Abbr" in tag
                or "Name" in tag or "Geox" in tag):
            abbreviations.append(low)
        else:
            emphatic.append(low)
    return abbreviations, emphatic


def numbers(text):
    return {int(x) for x in re.findall(r"\d+", str(text)) if int(x) >= 10}


def negated(text):
    return bool(re.search(r"\b(не|ни|нет|нельзя|никогда|никто|ничего|без)\b|\b(не|без)[а-яё]{3,}", str(text).lower()))


def content_words(text):
    return [w for w in tokens(text) if len(w) > 3]


def show(title, items, limit=10):
    print(f"\n{title}: {len(items)}")
    for uid, before, after, note in items[:limit]:
        print(f"    [{uid}] {note}")
        print(f"    было:  {str(before)[:130]}")
        print(f"    стало: {str(after)[:130]}")

## Сборка

In [54]:
results = pd.read_json(RUN_DIR / "results.jsonl", lines=True)

results_main = results
fix_path = GEN / "runs" / "multiline" / "results.jsonl"
if fix_path.exists():
    fix = pd.read_json(fix_path, lines=True).assign(round=4)
    results = pd.concat([results[~results.uid.isin(fix.uid)], fix], ignore_index=True)

work = pd.read_parquet(GEN / "work.parquet")

ok = results[results.status == "ok"].drop_duplicates(["uid", "level"], keep="last")
last = results.drop_duplicates(["uid", "level"], keep="last").set_index(["uid", "level"]).status
last.loc[list(zip(ok.uid, ok.level))] = "ok"

status = last.unstack("level").reindex(work.uid)
texts = ok.pivot(index="uid", columns="level", values="text").reindex(work.uid)


def reason(row):
    if (row == "ok").all():
        return "ok"
    if (row == "blocked").any():
        return "blocked"
    return next(s for s in row if s != "ok")


docs = work.set_index("uid")[["corpus", "id", "order", "text", "n_words"]].rename(columns={"text": "orig"})
docs[LEVELS] = texts[LEVELS]
docs[[f"{lv}_status" for lv in LEVELS]] = status[LEVELS].values
docs["status"] = status[LEVELS].apply(reason, axis=1)

print(f"пар документ-условие в журнале: {results[['uid', 'level']].drop_duplicates().shape[0]} из {len(work) * len(LEVELS)}")
print(f"строк журнала: {len(results)}, по кругам: {results.groupby('round').size().to_dict()}\n")

summary = pd.crosstab(docs.corpus, docs.status, margins=True, margins_name="всего")
print(summary.to_string())
print("\nдоля обработанных:")
print((summary["ok"] / summary["всего"]).map("{:.1%}".format).to_string())

пар документ-условие в журнале: 135000 из 135000
строк журнала: 138081, по кругам: {1: 130347, 2: 12, 3: 3069, 4: 4653}

status  blocked  length     ok  всего
corpus                               
rr           31       1  14968  15000
tg          197       3  14800  15000
yt          795       0  14205  15000
всего      1023       4  43973  45000

доля обработанных:
corpus
rr       99.8%
tg       98.7%
yt       94.7%
всего    97.7%


In [55]:
print("пары по условиям:")
print(pd.DataFrame({lv: docs[f"{lv}_status"].value_counts() for lv in LEVELS}).fillna(0).astype(int).to_string())

print("\nнеобработанные по длине исходного текста (слов), %:")
bins = [0, 5, 10, 15, 20, 25, 35]
docs["длина"] = pd.cut(docs.n_words, bins=bins)
lost = docs.assign(потерян=docs.status != "ok").groupby(["corpus", "длина"], observed=True).потерян
print((lost.mean().unstack("corpus") * 100).round(1).to_string())

for reason_name in sorted(set(docs.status) - {"ok", "blocked"}):
    part = docs[docs.status == reason_name]
    print(f"\n{reason_name}: {len(part)}")
    for uid, row in part.iterrows():
        print(f"    [{uid}] {row.n_words} слов: {str(row.orig)[:150]}")

пары по условиям:
         level1  level2  level3
ok        43978   43979   43975
blocked    1021    1020    1022
length        1       1       3

необработанные по длине исходного текста (слов), %:
corpus     rr   tg   yt
длина                  
(0, 5]    0.1  1.3  3.6
(5, 10]   0.2  1.5  4.1
(10, 15]  0.2  1.3  6.1
(15, 20]  0.2  0.8  7.0
(20, 25]  0.2  0.9  6.2
(25, 35]  0.3  1.3  7.5

length: 4
    [tg-93404] 16 слов: Почему такая ошибка. Может кто то сталкивался, все пакеты установил и переустановил, но проблема не решается.
(fastvenv) bobojon@user_unknown:~/FastAp
    [rr-22617] 24 слов: Заказала в августе, когда почти подошел срок - продлили еще на месяц, а потом еще на 2 месяца. Заказа так и не дождалась, деньги улетели. Твари.
    [tg-85169] 4 слов: заказ на 350р кому
    [tg-45289] 10 слов: могу попытаться - так, если получится платите, если нет - мне опыт


In [56]:
final = docs[docs.status == "ok"].reset_index()
columns = ["uid", "id", "orig"] + LEVELS

for name in CORPORA:
    part = final[final.corpus == name].sort_values("order")
    part[columns].to_parquet(FINAL / f"{name}.parquet", index=False)
    print(f"{name}: {len(part)} -> {name}.parquet")

docs.reset_index()[["uid", "corpus", "n_words"] + [f"{lv}_status" for lv in LEVELS] + ["status"]] \
    .to_parquet(FINAL / "status.parquet", index=False)

yt: 14205 -> yt.parquet
tg: 14800 -> tg.parquet
rr: 14968 -> rr.parquet


In [57]:
spend_log = pd.concat([results_main, fix]) if fix_path.exists() else results_main
spend_log["ts"] = pd.to_datetime(spend_log.ts)
spend = spend_log.groupby("round").agg(
    запросов=("uid", "size"), начало=("ts", "min"), конец=("ts", "max"),
    токенов=("tokens", "sum"), закэшировано=("precached_tokens", "sum"),
    секунд_ответа=("elapsed", "sum"))
spend["часов"] = ((spend.конец - spend.начало).dt.total_seconds() / 3600).round(1)
print(spend.to_string())
print(f"\nвсего токенов: {spend_log.tokens.sum() / 1e6:.1f} млн, "
      f"закэшировано: {spend_log.precached_tokens.sum() / 1e6:.1f} млн")
print(f"версии модели: {spend_log.model_version.dropna().unique().tolist()}")

       запросов              начало               конец   токенов  закэшировано  секунд_ответа  часов
round                                                                                                
1        135000 2026-11-07 19:28:04 2026-11-08 07:40:20  11004712      32671823       42652.21   12.2
2            13 2026-11-08 09:31:06 2026-11-08 09:33:58     50560          3305         172.57    0.0
3          3069 2026-11-08 09:58:03 2026-11-08 10:17:31    933021          2741        1148.12    0.3
4          4653 2026-11-08 13:51:31 2026-11-08 14:30:37    523686       1152457        2311.90    0.7

всего токенов: 12.5 млн, закэшировано: 33.8 млн
версии модели: ['GigaChat-3-Lightning:32.4.16.3']


## Описательные статистики

In [58]:
def n_words(text):
    return len(WORD.findall(str(text)))


def stats(texts):
    n_w = texts.map(n_words)
    q = [1, 5, 10, 25, 50, 75, 90, 95, 99]
    row = {"документов": len(texts), "медиана": int(n_w.median()), "среднее": round(n_w.mean(), 1),
           "минимум": int(n_w.min()), "максимум": int(n_w.max())}
    row.update({f"{p}%": int(v) for p, v in zip(q, np.percentile(n_w, q))})
    return row


rows = []
for name in CORPORA:
    part = final[final.corpus == name]
    for version in VERSIONS:
        rows.append({"корпус": name, "версия": version, **stats(part[version])})
print(pd.DataFrame(rows).to_string(index=False))

корпус версия  документов  медиана  среднее  минимум  максимум  1%  5%  10%  25%  50%  75%  90%  95%  99%
    yt   orig       14205       11     13.0        3        35   3   3    4    6   11   18   26   30   34
    yt level1       14205       11     13.0        0        36   3   3    4    6   11   18   26   30   34
    yt level2       14205       11     13.0        0        39   3   3    4    6   11   18   26   30   34
    yt level3       14205       13     15.2        0        77   3   4    5    7   13   21   30   34   43
    tg   orig       14800        8      9.6        3        35   3   3    3    5    8   12   19   23   32
    tg level1       14800        8      9.6        0        39   3   3    3    5    8   12   19   24   32
    tg level2       14800        8      9.6        0        68   3   3    3    5    8   12   19   23   32
    tg level3       14800        9     11.1        0       323   2   3    4    6    9   14   21   27   38
    rr   orig       14968       13     14.5   

In [59]:
rows = []
for name in CORPORA:
    part = final[final.corpus == name]
    for version in VERSIONS:
        forms = pd.Series([w for t in part[version] for w in tokens(t)])
        lems = forms.map(lemma)
        rows.append({"корпус": name, "версия": version, "словоупотреблений": len(forms),
                     "словоформ": forms.nunique(), "лемм": lems.nunique(),
                     "лемм с частотой от 5": int((lems.value_counts() >= 5).sum())})
print(pd.DataFrame(rows).to_string(index=False))

корпус версия  словоупотреблений  словоформ  лемм  лемм с частотой от 5
    yt   orig             184374      31603 17172                  3428
    yt level1             184322      30670 16133                  3443
    yt level2             183974      30137 15557                  3441
    yt level3             216175      28977 12335                  3719
    tg   orig             142118      24111 13515                  2634
    tg level1             142375      23729 13046                  2646
    tg level2             141456      23194 12573                  2619
    tg level3             163664      22640  9656                  2884
    rr   orig             217225      14019  7540                  2103
    rr level1             217139      13317  6857                  2084
    rr level2             216881      12941  6502                  2073
    rr level3             258496      12821  5522                  2188


## Проверки на полном наборе

In [60]:
pct = lambda x: f"{x:.1%}"
rows = []
for name in CORPORA:
    part = final[final.corpus == name]
    for level in LEVELS:
        pairs = list(zip(part.orig, part[level]))
        n = len(pairs)
        changed = [replacements(a, b) for a, b in pairs]
        rows.append({
            "корпус": name, "условие": level,
            "без изменений": pct(sum(" ".join(str(a).split()) == " ".join(str(b).split()) for a, b in pairs) / n),
            "правка написания": pct(sum(tokens(a) != tokens(b) and not r for (a, b), r in zip(pairs, changed)) / n),
            "замена слов": pct(sum(bool(r) for r in changed) / n),
            "прирост слов": pct(sum(len(content_words(b)) > len(content_words(a)) * 1.05 for a, b in pairs) / n),
            "утрата слов": pct(sum(len(content_words(b)) < len(content_words(a)) * 0.9 for a, b in pairs) / n),
            "отношение длин": round(float(np.median([len(tokens(b)) / max(len(tokens(a)), 1) for a, b in pairs])), 2)
        })
print(pd.DataFrame(rows).to_string(index=False))

корпус условие без изменений правка написания замена слов прирост слов утрата слов  отношение длин
    yt  level1         65.1%            15.3%        2.6%         1.2%        1.0%            1.00
    yt  level2         23.7%            23.9%        5.3%         2.7%        1.6%            1.00
    yt  level3          0.5%             4.7%       93.1%        72.0%        9.1%            1.14
    tg  level1         71.6%            12.1%        2.8%         1.4%        1.8%            1.00
    tg  level2         23.6%            20.6%        5.1%         2.7%        3.9%            1.00
    tg  level3          1.2%             6.1%       91.3%        65.6%       11.0%            1.09
    rr  level1         65.5%            11.5%        1.7%         0.7%        0.3%            1.00
    rr  level2         16.6%            21.9%        4.5%         2.5%        0.5%            1.00
    rr  level3          0.2%             1.6%       97.9%        78.6%        5.5%            1.20


In [61]:
rows = []
for name in CORPORA:
    part = final[final.corpus == name]
    for version in VERSIONS:
        texts = part[version]
        rows.append({
            "корпус": name, "версия": version,
            "аббревиатуры": pct(texts.map(lambda t: len(caps_split(t)[0]) > 0).mean()),
            "эмфатический капс": pct(texts.map(lambda t: len(caps_split(t)[1]) > 0).mean()),
            **{label: pct(texts.map(lambda t, p=PHENOMENA[label]: has(t, p)).mean())
               for label in ["эмодзи", "смайлы", "ряд знаков", "удлиненные"]},
        })
print(pd.DataFrame(rows).to_string(index=False))

корпус версия аббревиатуры эмфатический капс эмодзи смайлы ряд знаков удлиненные
    yt   orig         2.4%              3.1%  16.0%   3.8%      11.1%       1.4%
    yt level1         2.2%              3.0%  15.8%   3.8%      10.9%       1.2%
    yt level2         2.7%              2.4%  15.7%   3.6%      10.9%       1.1%
    yt level3         1.6%              0.3%   0.1%   0.0%       0.5%       0.1%
    tg   orig         0.6%              0.3%   2.0%   3.5%       2.5%       0.3%
    tg level1         0.5%              0.3%   1.9%   3.5%       2.4%       0.3%
    tg level2         1.2%              0.3%   1.9%   3.4%       2.4%       0.3%
    tg level3         1.2%              0.1%   0.0%   0.2%       0.3%       0.0%
    rr   orig         0.4%              1.4%   0.1%   5.5%      11.2%       0.9%
    rr level1         0.3%              1.4%   0.1%   5.4%      11.1%       0.8%
    rr level2         0.3%              1.1%   0.1%   5.1%      11.0%       0.5%
    rr level3         0.2%  

In [62]:
addressed = lambda t: has(t, PHENOMENA["обращение на ты"])
question = lambda t: has(t, PHENOMENA["вопрос"])


def share(pairs, condition, lost):
    base = [(a, b) for a, b in pairs if condition(a)]
    return f"{sum(lost(a, b) for a, b in base) / max(len(base), 1):.1%} из {len(base)}"


rows = []
for name in CORPORA:
    part = final[final.corpus == name]
    for level in LEVELS:
        pairs = list(zip(part.orig, part[level]))
        rows.append({
            "корпус": name, "условие": level,
            "утрачены числа": share(pairs, numbers, lambda a, b: bool(numbers(a) - numbers(b))),
            "утрачено отрицание": share(pairs, negated, lambda a, b: not negated(b)),
            "смена обращения": share(pairs, addressed, lambda a, b: not addressed(b)),
            "вопрос стал утверждением": share(pairs, question, lambda a, b: not question(b)),
            "пересказ со стороны": int(part[level].str.contains(
                r"\b(?:автор|говорящий|пишущий)\s+(?:пишет|отмечает|указывает|считает)", regex=True).sum())
        })
print(pd.DataFrame(rows).to_string(index=False))

корпус условие утрачены числа утрачено отрицание смена обращения вопрос стал утверждением  пересказ со стороны
    yt  level1   0.5% из 1879       0.1% из 5798     0.6% из 712             0.6% из 2065                    0
    yt  level2   0.9% из 1879       0.2% из 5798     1.4% из 712             0.2% из 2065                    0
    yt  level3  31.6% из 1879      13.1% из 5798    76.4% из 712            42.4% из 2065                    0
    tg  level1    1.6% из 855       0.1% из 4917    0.8% из 1060             1.2% из 2483                    0
    tg  level2    1.5% из 855       0.2% из 4917    0.9% из 1060             0.6% из 2483                    0
    tg  level3   18.1% из 855      14.5% из 4917   80.3% из 1060            31.2% из 2483                    0
    rr  level1   0.1% из 3667       0.0% из 9242       0.0% из 9              2.9% из 102                    0
    rr  level2   0.4% из 3667       0.0% из 9242       0.0% из 9              2.0% из 102                    0
 

## Фильтр длины

In [69]:
import re
import pandas as pd

from src.paths import DATA

FINAL = DATA / "generation" / "final"
LEVELS = ["level1", "level2", "level3"]
CORPORA = ["yt", "tg", "rr"]

WORD = re.compile(r"[А-Яа-яЁё]+")
n_words = lambda text: len(WORD.findall(str(text)))

final = pd.concat([pd.read_parquet(FINAL / f"{c}.parquet").assign(corpus=c) for c in CORPORA],
                  ignore_index=True)
print(f"документов: {len(final)}")

документов: 43973


In [70]:
UP_12 = 2      # условия 1 и 2: результат длиннее исходного больше чем вдвое
UP_3 = 3       # условие 3: длиннее больше чем втрое
DOWN = 1 / 3   # любое условие: короче больше чем втрое

orig = final.orig.map(n_words).clip(lower=1)
ratio = pd.DataFrame({lv: final[lv].map(n_words) / orig for lv in LEVELS})
upper = pd.Series({"level1": UP_12, "level2": UP_12, "level3": UP_3})

too_long = ratio.gt(upper)
too_short = ratio.lt(DOWN)
drop = (too_long | too_short).any(axis=1)

print("документов по условиям:")
print(pd.DataFrame({"длиннее": too_long.sum(), "короче": too_short.sum(),
                    "всего": (too_long | too_short).sum()}).to_string())

by_corpus = pd.DataFrame({"было": final.groupby("corpus").size(),
                          "удаляется": drop.groupby(final.corpus).sum()}).reindex(CORPORA)
by_corpus["доля"] = (by_corpus["удаляется"] / by_corpus["было"]).map("{:.2%}".format)
print(f"\nудаляется документов: {drop.sum()} ({drop.mean():.2%})")
print(by_corpus.to_string())

short = final.orig.map(n_words) <= 5
print(f"\nкоротких исходников (до 5 слов): в корпусе {short.mean():.1%}, среди удаляемых {short[drop].mean():.1%}")

документов по условиям:
        длиннее  короче  всего
level1       16      17     33
level2        5      18     23
level3      331      24    355

удаляется документов: 377 (0.86%)
         было  удаляется   доля
corpus                         
yt      14205        124  0.87%
tg      14800        241  1.63%
rr      14968         12  0.08%

коротких исходников (до 5 слов): в корпусе 21.2%, среди удаляемых 70.3%


In [71]:
kept = final[~drop]
for c in CORPORA:
    part = kept[kept.corpus == c].drop(columns="corpus")
    part.to_parquet(FINAL / f"{c}.parquet", index=False)
    print(f"{c}: {len(part)}")

status = pd.read_parquet(FINAL / "status.parquet")
status.loc[status.uid.isin(final.uid[drop]), "status"] = "length_ratio"
status.to_parquet(FINAL / "status.parquet", index=False)
print("\n" + status.status.value_counts().to_string())

yt: 14081
tg: 14559
rr: 14956

status
ok              43596
blocked          1023
length_ratio      377
length              4
